<a href="https://colab.research.google.com/github/JariusF10-8/bt4222/blob/main/notebooks/Dataset_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MIND-small: Data Preprocessing

## Setup

In [10]:
import pandas as pd
import numpy as np
import csv
import os
import json
import re
from pathlib import Path
from google.colab import drive


drive.mount("/content/drive")

DATA_ROOT = Path("/content/drive/MyDrive/BT4222 Group Project/Microsoft MIND")

BEHAVIORS_COLUMNS = [
    "impression_id", "user_id", "time", "history", "impressions"
]

NEWS_COLUMNS = [
    "news_id", "category", "subcategory", "title", "abstract",
    "url", "title_entities", "abstract_entities"
]


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Data Sampling & Splitting

### Subsetting and Sampling Approach

The full training partition is retained; no additional event sampling is performed. Validation and test always retain every candidate and label. Negative sampling, if later needed, should be a separate training-only experiment using non-clicked candidates from the same event; it changes the class balance and does not produce calibrated click probabilities automatically.


In [11]:
TRAIN_START = pd.Timestamp("2019-11-09")
VALIDATION_START = pd.Timestamp("2019-11-13")  # Inclusive lower bound
TRAIN_FILE_END = pd.Timestamp("2019-11-15")  # Exclusive upper bound

# Reuse the column definitions from Setup.
SAMPLING_BEHAVIOR_COLUMNS = BEHAVIORS_COLUMNS
SAMPLING_NEWS_COLUMNS = NEWS_COLUMNS


def require(condition, message):
    if not condition:
        raise ValueError(message)


def read_sampling_tsv(path, columns):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Required MIND file not found: {path}")
    return pd.read_csv(
        path, sep="\t", header=None, names=columns, dtype="string",
        keep_default_na=False, na_values=[""], quoting=csv.QUOTE_NONE,
        encoding="utf-8",
    )

# prepare events and candidates for modeling, with validation checks
def prepare_events(raw, source_file):
    events = raw.copy()
    require(len(events) > 0, f"No events in {source_file}")
    for column in ["impression_id", "user_id", "time", "impressions"]:
        require(
            events[column].fillna("").str.strip().ne("").all(),
            f"Missing {column} in {source_file}; resolve in Section 3.2.",
        )
    require(
        events["impression_id"].is_unique,
        f"Repeated impression IDs in {source_file}; review before splitting.",
    )
    events["timestamp"] = pd.to_datetime(
        events["time"], format="%m/%d/%Y %I:%M:%S %p", errors="raise"
    )
    require(events["timestamp"].notna().all(), "Missing parsed timestamps")
    events["source_file"] = source_file
    events["event_key"] = list(zip(events["source_file"], events["impression_id"]))
    events["history_ids"] = events["history"].fillna("").str.split()
    events["history_missing"] = events["history_ids"].str.len().eq(0)
    # Recorded times have no documented timezone: do not assume local time.
    return events.sort_values(
        ["timestamp", "impression_id"], kind="stable"
    ).reset_index(drop=True)


def expand_candidates(events):
    # Compact table: do not repeat titles or entire histories per candidate.
    tokens = events["impressions"].str.split().explode()
    valid = tokens.str.fullmatch(r"N[0-9]+-[01]", na=False)
    require(valid.all(), f"Malformed candidate tokens: {tokens[~valid].head().tolist()}")
    parts = tokens.str.rsplit("-", n=1, expand=True)
    candidates = pd.DataFrame({
        "event_key": events.loc[tokens.index, "event_key"].to_numpy(),
        "candidate_position": tokens.groupby(level=0, sort=False).cumcount().to_numpy(),
        "news_id": parts[0].to_numpy(),
        "label": parts[1].astype("int8").to_numpy(),
    })
    candidates["news_id"] = candidates["news_id"].astype("string")
    candidates["candidate_position"] = candidates["candidate_position"].astype("int32")
    return candidates

print(f"Training window: {TRAIN_START.date()} <= time < {VALIDATION_START.date()}")
print(f"Validation window: {VALIDATION_START.date()} <= time < {TRAIN_FILE_END.date()}")


Training window: 2019-11-09 <= time < 2019-11-13
Validation window: 2019-11-13 <= time < 2019-11-15


### Temporal Split Strategy & Data Leakage Prevention

| Project partition | Source and recorded date | Purpose |
|---|---|---|
| Training | `MINDsmall_train`, November 9–12, 2019 | Fit preprocessing and models |
| Validation | `MINDsmall_train`, November 13–14, 2019 | Select features, models and hyperparameters |
| Final test | Entire `MINDsmall_dev`, verified to occur later | Evaluate the fixed model once |



The same user or article may occur in multiple partitions: our task is to predict future interactions, not exclusively interactions from unseen users. Keep unseen users/articles too and use fallback features. For the initial experiment, keep the chosen model fitted on November 9–12; do not refit on validation before testing.


**Data leakage prevention implemented in the split cell below**

The following steps separate earlier training events from later evaluation events. The helper functions called here are defined in the Section 3.3 code cell.

| Step | Code to look for in the split cell | What it prevents or preserves |
|---|---|---|
| 1. Load explicitly named sources separately | `for source in ["MINDsmall_train", "MINDsmall_dev"]`, followed by `original_train_events` and `original_test_events` | Avoids accidentally treating the development file as training because the overview's `SPLIT` setting changed. Development events are assigned only to the project test partition. |
| 2. Preserve each event's supplied prior history | `prepare_events(...)` | The helper parses the supplied `history` into `history_ids`; it does not combine a user's clicks across the dataset or fill empty histories with future clicks. It relies on the supplied histories being prior histories; individual history-click timestamps are unavailable for independent verification. |
| 3. Validate dates and split whole events by timestamp | The training-source date-range `require(...)` and `events_by_partition` | Requires the training source to fall within November 9–14. Events before November 13 enter training; November 13–14 events enter validation. File order, impression ID order and random candidate-row assignment do not determine the split. |
| 4. Enforce strict chronological separation | The two `require(...)` checks comparing `.max()` and `.min()` timestamps | Stops unless the latest training event precedes the earliest validation event, and the latest validation event precedes the earliest test event. Thus no timestamp is shared across adjacent partitions. |
| 5. Check that event keys do not repeat across partitions | `len(all_event_keys) == len(set(all_event_keys))` | Checks that no `(source_file, impression_id)` key is repeated across the combined partitions. This checks event identity by key, not duplicate content stored under different keys. |
| 6. Expand candidates only after assigning events | `candidates_by_partition = {name: expand_candidates(events) ...}` | All candidates from an event inherit the same partition. No candidate from a training event is separately assigned to validation or test. The helper extracts click labels as a separate column, but later feature code must still exclude that column from inputs. |

**Supporting integrity checks:** The nonempty-partition checks and total-event-count check stop an empty or incomplete split. These protect data integrity; they do not independently prevent feature leakage. The called helpers also reject missing required fields, malformed timestamps/candidate labels and repeated impression IDs within a source.




In [12]:
# Read explicitly named sources; do not rely on the overview selected SPLIT.
source_events = {}
source_news = {}
for source in ["MINDsmall_train", "MINDsmall_dev"]:
    folder = Path(DATA_ROOT) / source
    source_events[source] = prepare_events(
        read_sampling_tsv(folder / "behaviors.tsv", SAMPLING_BEHAVIOR_COLUMNS),
        f"{source}/behaviors.tsv",
    )
    source_news[source] = read_sampling_tsv(
        folder / "news.tsv", SAMPLING_NEWS_COLUMNS
    )

original_train_events = source_events["MINDsmall_train"]
original_test_events = source_events["MINDsmall_dev"]
require(
    original_train_events["timestamp"].ge(TRAIN_START).all()
    and original_train_events["timestamp"].lt(TRAIN_FILE_END).all(),
    "Training source contains dates outside November 9–14; review the split policy.",
)

events_by_partition = {
    "train": original_train_events.loc[
        original_train_events["timestamp"].lt(VALIDATION_START)
    ].copy().reset_index(drop=True),
    "validation": original_train_events.loc[
        original_train_events["timestamp"].ge(VALIDATION_START)
    ].copy().reset_index(drop=True),
    "test": original_test_events.copy().reset_index(drop=True),
}
for name, events in events_by_partition.items():
    require(len(events) > 0, f"Empty {name} partition")

require(
    events_by_partition["train"]["timestamp"].max()
    < events_by_partition["validation"]["timestamp"].min(),
    "Training and validation times overlap.",
)
require(
    events_by_partition["validation"]["timestamp"].max()
    < events_by_partition["test"]["timestamp"].min(),
    "Development events are not strictly later than validation; stop and review.",
)

all_event_keys = [
    key for events in events_by_partition.values() for key in events["event_key"]
]
require(len(all_event_keys) == len(set(all_event_keys)), "An event crosses partitions.")
require(
    len(all_event_keys) == len(original_train_events) + len(original_test_events),
    "The split lost or duplicated events.",
)

candidates_by_partition = {
    name: expand_candidates(events) for name, events in events_by_partition.items()
}

print("Temporal split passed. Main experiments use the full training partition.")

Temporal split passed. Main experiments use the full training partition.


**Retains metadata for history and candidate articles, handles missing metadata, rejects duplicate article IDs, and preserves candidate counts.**

Each partition uses its own source file's article metadata; held-out metadata never fills training gaps. Keep one row for every article referenced by a history or candidate list. 

Missing metadata receives empty text/entity lists and `__UNKNOWN__` categories, plus `metadata_missing=True`. Missing abstracts and other individual fields remain available for Section 3.2 handling. The left-join check below verifies that missing metadata does not drop candidates and duplicate metadata does not multiply them.

Candidate tables stay compact; join article features by `news_id` with `how="left", validate="many_to_one"` when modelling. Fit text preprocessing only on `news_by_partition["train"]` (excluding missing metadata). Transform validation/test content with those fitted objects. Do not concatenate article collections across partitions to fit a vocabulary or TF-IDF.

In [13]:
def referenced_news_ids(events, candidates):
    history_ids = {article for history in events["history_ids"] for article in history}
    return history_ids | set(candidates["news_id"])


def retain_metadata(events, candidates, metadata):
    require(
        metadata["news_id"].fillna("").str.strip().ne("").all(),
        "Missing article IDs in metadata; review Section 3.2.",
    )
    require(metadata["news_id"].is_unique, "Duplicate article IDs would multiply joins.")
    referenced_ids = referenced_news_ids(events, candidates)
    required = pd.DataFrame({"news_id": pd.Series(sorted(referenced_ids), dtype="string")})
    retained = required.merge(
        metadata, on="news_id", how="left", validate="one_to_one", indicator=True
    )
    retained["metadata_missing"] = retained.pop("_merge").eq("left_only")
    absent = retained["metadata_missing"]
    for column in ["title", "abstract", "url"]:
        retained.loc[absent, column] = ""
    for column in ["category", "subcategory"]:
        retained.loc[absent, column] = "__UNKNOWN__"
    for column in ["title_entities", "abstract_entities"]:
        retained.loc[absent, column] = "[]"

    # Only join the coverage flag here, avoiding repeated text for millions of rows.
    coverage = candidates[["news_id"]].merge(
        retained[["news_id", "metadata_missing"]], on="news_id", how="left",
        validate="many_to_one", sort=False,
    )
    require(len(coverage) == len(candidates), "Article join changed candidate count.")
    require(coverage["metadata_missing"].notna().all(), "Unaccounted candidate metadata.")
    audit = {
        "referenced_articles": len(retained),
        "articles_missing_metadata": int(absent.sum()),
        "candidate_exposures_missing_metadata": int(coverage["metadata_missing"].sum()),
    }
    return retained, audit


news_by_partition = {}
metadata_audits = {}
for name, events in events_by_partition.items():
    source = "MINDsmall_dev" if name == "test" else "MINDsmall_train"
    news_by_partition[name], metadata_audits[name] = retain_metadata(
        events, candidates_by_partition[name], source_news[source]
    )


**Check that candidate data stay intact and display summary of partitions**

In [14]:
def verify_candidate_integrity(events, candidates):
    require(
        not candidates.duplicated(["event_key", "candidate_position"]).any(),
        "Repeated candidate positions within an event.",
    )
    require(
        set(candidates["event_key"]) == set(events["event_key"]),
        "Candidate table lost an event or introduced an unknown event.",
    )
    # Reconstruct complete token sequences: verifies IDs, labels, order and counts.
    ordered = candidates.sort_values("candidate_position", kind="stable")
    tokens = ordered["news_id"] + "-" + ordered["label"].astype("string")
    reconstructed = tokens.groupby(ordered["event_key"], sort=False).agg(" ".join)
    expected = events.set_index("event_key")["impressions"].str.split().str.join(" ")
    require(
        reconstructed.reindex(expected.index).eq(expected).all(),
        "Candidate IDs or labels changed during splitting/expansion/sampling.",
    )


summary_rows = []
for name, events in events_by_partition.items():
    candidates = candidates_by_partition[name]
    verify_candidate_integrity(events, candidates)
    clicks = int(candidates["label"].sum())
    summary_rows.append({
        "partition": name,
        "earliest_time": events["timestamp"].min(),
        "latest_time": events["timestamp"].max(),
        "events": len(events),
        "users": events["user_id"].nunique(),
        "candidate_exposures": len(candidates),
        "clicked_candidates": clicks,
        "nonclicked_candidates": len(candidates) - clicks,
        "events_with_empty_history": int(events["history_missing"].sum()),
        **metadata_audits[name],
    })
split_summary = pd.DataFrame(summary_rows).set_index("partition")
display(split_summary)
print("Passed: chronology, event separation, complete candidate labels, and metadata joins.")

,earliest_time,latest_time,events,users,candidate_exposures,clicked_candidates,nonclicked_candidates,events_with_empty_history,referenced_articles,articles_missing_metadata,candidate_exposures_missing_metadata
partition,,,,,,,,,,,
train,2019-11-09 00:00:19,2019-11-12 23:59:25,95071,40148,3389342,141558,3247784,1735,43370,0,0
validation,2019-11-13 00:01:00,2019-11-14 23:59:13,61894,31664,2454102,94786,2359316,1503,38421,0,0
test,2019-11-15 00:00:01,2019-11-15 23:58:03,73152,50000,2740998,111383,2629615,2214,42416,0,0


Passed: chronology, event separation, complete candidate labels, and metadata joins.


In [15]:
partition_sizes = split_summary.loc[ ["train", "validation"], ["events"]].copy()

partition_sizes["percentage_of_original_train"] = (
    partition_sizes["events"] / partition_sizes["events"].sum() * 100).round(2)

display(partition_sizes)

,events,percentage_of_original_train
partition,,
train,95071,60.57
validation,61894,39.43
